# Teksta apstrāde ar Python: darba burtnīca

Šī ir vadīta darba burtnīca 2. dienai. Tā saglabā pilnā notebook galveno struktūru, bet nesatur gatavus risinājumus.

Darba pieeja:
- izlasi sadaļas mērķi;
- aizpildi `TODO` vietas;
- palaid sekcijas pa vienai;
- salīdzini rezultātus ar sagaidāmo izvadi.


## Neapstrādāta teksta faila pārbaude

Mērķis: pārbaudīt neapstrādātu teksta failu un izveidot īsu kvalitātes pārskatu.

Ievades fails:
- `data/day2/responses_raw.txt`

Sagaidāmā izvade:
- Python objekts: `dict[str, object]`
- Konsolē izdrukāts pārskats par tukšajām rindām, cipariem, dubultatstarpēm un dublikātiem


In [1]:
from collections import Counter
from pathlib import Path # Path biblioteka ir iebūveta darbam ar mapēm un failiem
import csv
import re
import string

# lieliem burtiem rakstītie mainīgie parasti tiek izmantoti kā konstantes,
#  tas ir, vērtības, kas nemainās programmas izpildes laikā. Šajā gadījumā mēs izmantojam lielus burtus, lai norādītu, ka šie mainīgie ir ceļi uz datu failiem, un tie netiks mainīti programmas izpildes laikā. Tas ir arī labs veids, kā organizēt un strukturēt kodu, jo tas ļauj viegli atrast un mainīt šos ceļus, ja nepieciešams.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DAY2_DATA_DIR = PROJECT_ROOT / 'data' / 'day2'

RAW_PATH = DAY2_DATA_DIR / 'responses_raw.txt'
STOPWORDS_PATH = DAY2_DATA_DIR / 'stopwords_lv.txt'
GENERATED_CLEANED_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses.txt'
GENERATED_FREQ_PATH = DAY2_DATA_DIR / 'generated_word_frequencies.txt'
GENERATED_CLEANED_V2_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses_v2.txt'
GENERATED_FREQ_V2_PATH = DAY2_DATA_DIR / 'generated_word_frequencies_v2.txt'
GENERATED_CLEANED_FINAL_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses_final.txt'
GENERATED_FREQ_FINAL_PATH = DAY2_DATA_DIR / 'generated_word_frequencies_final.txt'
SURVEYS_FOLDER = DAY2_DATA_DIR / 'surveys'
SURVEY_OUTPUT_PATH = SURVEYS_FOLDER / 'survey_responses_summary.csv'

print(PROJECT_ROOT)
print(DAY2_DATA_DIR)


D:\Github\RTU_Python_CSP
D:\Github\RTU_Python_CSP\data\day2


In [ ]:
# datora mums ir divu veidu ceļi uz failiem, absolūtais ceļš un relatīvais ceļš. 
# Absolūtais ceļš sākas no saknes direktorijas un iet cauri visām mapēm līdz konkrētajam failam, 
# piemēram, C:\Users\Username\Documents\file.txt.
#  Relatīvais ceļš ir attiecībā pret pašreizējo darba direktoriju,
#  un tas var būt īsāks un ērtāks, piemēram, data/file.txt, ja pašreizējā darba direktorija ir C:\Users\Username\Documents. Relatīvie ceļi ir elastīgāki un portablāki, jo tie nav atkarīgi no konkrētas datora struktūras, un tie ļauj mums viegli pārvietot mūsu kodu un datus uz citiem datoriem vai mapēm bez nepieciešamības mainīt ceļus.
# piemēram notebook man relatīvi atrodas: notebooks - tātad pret projekta sakni
# bet absolutais notebook mape man konkrēti ir D:\Github\RTU_Python_CSP\notebooks

# ja strādājam vieni paši vienā datora tad absolutais ir vienkāršāks, 
# bet ja strādājam komandā vai vēlamies, lai mūsu kods būtu portabls
#  un varētu darboties uz dažādiem datoriem, tad relatīvais ceļš ir labāks risinājums. Relatīvie ceļi ļauj mums viegli pārvietot mūsu kodu un datus uz citiem datoriem vai mapēm bez nepieciešamības mainīt ceļus, un tas arī padara mūsu kodu vieglāk lasāmu un saprotamu, jo mēs varam izmantot vienkāršākus un īsākus ceļus, kas attiecas uz mūsu projekta struktūru.

In [2]:
# tātad ja mēs zinam kur dzīvo fails tad mēs varam to nolasīt ar open funkciju, un tad varam izmantot šo faila objektu, lai lasītu tā saturu. Piemēram:
# relative_path = r'data\day2\survey_response.txt'
# Windows izmanto \ ceļu atdalīšanai, bet Python interpretē \ kā speciālu simbolu, tāpēc mums ir jāizmanto r'...' sintakse, lai norādītu, ka šis ir raw string, un \ tiek interpretēts kā parasts simbols. Alternatīvi, mēs varam izmantot / ceļu atdalīšanai, kas darbojas gan Windows, gan Unix sistēmās, un Python to pareizi interpretēs. Piemēram:
# relative_path = 'data/day2/survey_response.txt'
# why es varēšu nolasīt šo failu ar relatīvo ceļu?
# šis notebook relatīvais ceļš ir
# notebooks\day2_text_processing_clean_workbook.ipynb
# so I am in notebooks but data is in data - a SIBLING folder
# so actual relative path would start one level up
relative_path = '../data/day2/survey_response.txt' # ceļs šeit ir tikai vienkāršs teksts
print(f"Relatīvais ceļš uz failu ir: {relative_path}")
# pārvertīsim šo relatīvo cēlu Path objektā, lai varētu izmantot tā metodes un īpašības
relative_path_obj = Path(relative_path)
print(f"Relatīvais ceļš kā Path objekts: {relative_path_obj}")
# tagad varu pārbaudīt vai šādas fails eksistē
if relative_path_obj.exists():
    print(f"Fails {relative_path_obj} eksistē.")
else:
    print(f"Fails {relative_path_obj} neeksistē.")
# ja fails eksistē tad varam to nolasīt ar open funkciju izmantojot šo relatīvo ceļu abi varianti

Relatīvais ceļš uz failu ir: ../data/day2/survey_response.txt
Relatīvais ceļš kā Path objekts: ..\data\day2\survey_response.txt
Fails ..\data\day2\survey_response.txt eksistē.


In [3]:
# tātad open atver mums failu - ta sauktaja straumēšanas režīmā, un mēs varam izmantot šo faila objektu, lai lasītu tā saturu. Piemēram:
# labā prakse ir izmanot with konteksta pārvaldnieku, lai atvērtu failu, jo tas nodrošina, ka fails tiks pareizi aizvērts pēc tam, kad mēs esam pabeiguši ar to strādāt, pat ja rodas kāda kļūda vai izņēmums. Ar with konteksta pārvaldnieku mēs varam būt droši, ka fails tiks aizvērts, kad mēs iziesim no with bloka, un tas arī padara mūsu kodu tīrāku un vieglāk lasāmu. Piemēram:
# ja nelietotu with tad būtu ar roku jāraksta file.close() pēc tam, kad esam pabeiguši ar failu strādāt, un ja aizmirstam to izdarīt, tad fails var palikt atvērts un radīt problēmas, piemēram, resursu noplūdi vai piekļuves kļūdas. Ar with konteksta pārvaldnieku mēs varam izvairīties no šādām problēmām un nodrošināt, ka mūsu kods ir drošs un efektīvs.
with open(relative_path, 'r', encoding='utf-8') as file: # atkal kols tātad atkāpe
    # file šeit ir atvērts, file ir vienkārs mainīgais, kas satur faila objektu, un mēs varam izmantot šo objektu, lai lasītu tā saturu. Piemēram, mēs varam izmantot file.read() metodi, lai nolasītu visu faila saturu kā vienu lielu tekstu. Šajā gadījumā mēs saglabājam šo tekstu mainīgajā content.
    content = file.read() # content ir vienkāršs mainīgais, kas satur visu faila saturu kā vienu lielu tekstu. Mēs varam izmantot šo mainīgo, lai veiktu dažādas teksta apstrādes operācijas, piemēram, meklēšanu, aizvietošanu, sadalīšanu un tā tālāk. Piemēram, mēs varam izmantot content.splitlines() metodi, lai sadalītu tekstu rindās un iegūtu sarakstu ar katru rindu kā atsevišķu elementu.
    # fails vel šeit vaļā
# fails šeit ir ciet
print(f"Fails nolasīts ar relatīvo ceļu. Saturs:\n{content}")

Fails nolasīts ar relatīvo ceļu. Saturs:
1. What is your name? This field was optional for respondents.
Jānis Berzins.

2. What is your age group?
I am 45 years old and have been living and working in Riga for most of my adult life.

3. What is your gender? This field was optional for respondents.
Male.

4. What contact details could be used for a follow-up survey?
For a follow-up survey, you could use the email janis.berzins45@example.com and the phone number +371 55550100.

5. What is your current employment situation?
I work full time as a project manager in a private company and I have been in a stable professional role for 12 years.

6. What is the highest level of education you have completed?
I have a master's degree in economics from a university in Latvia, which helped me build my career in business and finance related work.

7. How would you describe your household?
I live with my spouse and one teenage child in Riga, and we try to balance daily expenses with saving for educa

In [4]:
# kā nolasīt ar absolūto ceļu? Absolūtais ceļš ir konkrēts un sākas no saknes direktorijas, tāpēc mums ir jānorāda pilns ceļš līdz failam. Piemēram, ja mūsu fails atrodas D:\Github\RTU_Python_CSP\data\day2\survey_response.txt, tad absolūtais ceļš būtu 'D:/Github/RTU_Python_CSP/data/day2/survey_response.txt' vai 'D:\\Github\\RTU_Python_CSP\\data\\day2\\survey_response.txt'. Mēs varam izmantot šo absolūto ceļu ar open funkciju tāpat kā ar relatīvo ceļu. Piemēram:
absolute_path = 'D:/Github/RTU_Python_CSP/data/day2/survey_response.txt'
# tātad citiem šis absolūtais ceļš varētu būt atšķirīgs, atkarībā no tā, kur viņi ir saglabājuši šo failu savā datorā. Absolūtais ceļš ir konkrēts un sākas no saknes direktorijas, tāpēc tas var atšķirties no datora uz datoru. Relatīvais ceļš ir attiecībā pret pašreizējo darba direktoriju, un tas var būt īsāks un ērtāks, ja mēs zinām, kur atrodas mūsu fails attiecībā pret mūsu kodu. Tāpēc relatīvais ceļš ir biežāk izmantots un ieteicams, jo tas padara mūsu kodu portablāku un vieglāk lasāmu.
with open(absolute_path, 'r', encoding='utf-8') as file:
    content_absolute = file.read()
print(f"Fails nolasīts ar absolūto ceļu. Saturs:\n{content_absolute}")

Fails nolasīts ar absolūto ceļu. Saturs:
1. What is your name? This field was optional for respondents.
Jānis Berzins.

2. What is your age group?
I am 45 years old and have been living and working in Riga for most of my adult life.

3. What is your gender? This field was optional for respondents.
Male.

4. What contact details could be used for a follow-up survey?
For a follow-up survey, you could use the email janis.berzins45@example.com and the phone number +371 55550100.

5. What is your current employment situation?
I work full time as a project manager in a private company and I have been in a stable professional role for 12 years.

6. What is the highest level of education you have completed?
I have a master's degree in economics from a university in Latvia, which helped me build my career in business and finance related work.

7. How would you describe your household?
I live with my spouse and one teenage child in Riga, and we try to balance daily expenses with saving for educa

In [5]:
# pamēģinašim paši nolasīt latviešu versiju šim failam ar relatīvo ceļu, un tad ar absolūto ceļu, lai redzētu vai tas strādā un kāds ir rezultāts. Ja viss ir pareizi, tad mēs varētu redzēt, ka abi veidi nolasīs to pašu saturu un izvadīs to uz ekrāna. Ja rodas kāda kļūda, piemēram, fails netiek atrasts vai nav piekļuves tiesību, tad mēs varam saņemt atbilstošu kļūdas ziņojumu, kas palīdzēs mums diagnosticēt un novērst problēmu.
latvian_relative_path = r"..\data\day2\survey_response_lv.txt" # r nozīme raw string, lai \ tiktu interpretēts kā parasts simbols
with open(latvian_relative_path, 'r', encoding='utf-8') as file:
    content_lv = file.read()
print(f"Latviešu fails nolasīts ar relatīvo ceļu. Saturs:\n{content_lv}")

Latviešu fails nolasīts ar relatīvo ceļu. Saturs:
1. Kāds ir jūsu vārds? Šis lauks respondentiem bija neobligāts.
Jānis Berzins.

2. Kādā vecuma grupā jūs esat?
Man ir 45 gadi, un es lielāko daļu savas pieaugušā dzīves esmu dzīvojis un strādājis Rīgā.

3. Kāds ir jūsu dzimums? Šis lauks respondentiem bija neobligāts.
Vīrietis.

4. Kādus kontaktatus varētu izmantot turpmākai aptaujai?
Turpmākai aptaujai varētu izmantot e-pasta adresi janis.berzins45@example.com un tālruņa numuru +371 55550100.

5. Kāda ir jūsu pašreizējā nodarbinātības situācija?
Es strādāju pilnu slodzi kā projektu vadītājs privātā uzņēmumā, un jau 12 gadus man ir stabils profesionāls amats.

6. Kāds ir augstākais izglītības līmenis, ko esat ieguvis?
Man ir maģistra grāds ekonomikā Latvijas universitātē, un tas man palīdzēja izveidot karjeru ar biznesu un finansēm saistītā darbā.

7. Kā jūs raksturotu savu mājsaimniecību?
Es dzīvoju Rīgā kopā ar dzīvesbiedru un vienu pusaudzi, un mēs cenšamies līdzsvarot ikdienas izdev

In [ ]:
# Python supports other encodings 
# by default encoding for open is ASCII - English
# Usually we want UTF-8 encoding for text files
# Python supports many other encodings
# , which supports all languages and special characters.
# full list of supported encodings can be found here: https://docs.python.org/3/library/codecs.html#standard-encodings

In [ ]:



def split_into_lines(text: str) -> list[str]:
    """TODO: sadalīt tekstu rindās."""
    # HINT: str.splitlines()
    raise NotImplementedError


def find_blank_lines(lines: list[str]) -> list[int]:
    """TODO: atrast tukšo rindu numurus."""
    raise NotImplementedError


def find_digit_lines(lines: list[str]) -> list[int]:
    """TODO: atrast rindas, kurās ir cipari."""
    raise NotImplementedError


def find_lines_with_extra_spaces(lines: list[str]) -> list[int]:
    """TODO: atrast rindas ar dubultatstarpēm."""
    raise NotImplementedError


def find_duplicate_lines(lines: list[str]) -> list[tuple[str, int]]:
    """TODO: atrast dublētas netukšas rindas un to skaitu."""
    # HINT: Counter(...)
    raise NotImplementedError


def print_inspection_report(lines: list[str]) -> None:
    """TODO: izdrukāt kompaktu pārskatu."""
    raise NotImplementedError


In [6]:
def read_text_file(path: Path, encoding: str = 'utf-8') -> str:
    """nolasam visu faila saturu kā vienu virkni."""
    # veidojam Path objektu katram gadijumam, jo path var būt string
    path = Path(path) # ja padots path tad nekas nenotiks
    # pārbaudam vai eksistē fails
    if not path.exists():
        raise FileNotFoundError(f"Fails {path} netika atrasts.")
    # šeit zinam ka fails eksistē
    with open(path, 'r', encoding=encoding) as file:
        content = file.read() 
        # uzreiz neatgriežam jo fails varētu drusku ilgāk "pakarāties" vaļā
    # fails ir ciet  
    return content

# pārbaudam ar mūsu latviešu failu
content_lv = read_text_file(latvian_relative_path)
print(f"Latviešu fails nolasīts ar read_text_file funkciju. Satura sākums:\n{content_lv[:100]}")

Latviešu fails nolasīts ar read_text_file funkciju. Satura sākums:
1. Kāds ir jūsu vārds? Šis lauks respondentiem bija neobligāts.
Jānis Berzins.

2. Kādā vecuma grupā


In [7]:
# otrs veids kā strādat būtu nolasīt visu pa rindām
def read_text_file_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    """nolasam visu faila saturu kā sarakstu ar rindām."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Fails {path} netika atrasts.")
    with open(path, 'r', encoding=encoding) as file:
        lines = file.readlines() # nolasu visas rindas kā sarakstu, kur katra rinda ir atsevišķs elements
    return lines

latvian_lines = read_text_file_lines(latvian_relative_path)
# izdrukāsim pirmās 5 rindas
print("Pirmās 5 rindas no latviešu faila:")
for line in latvian_lines[:5]:
    print(line, end='') # end='' lai izvairītos no dubultām jauna rinda, jo line jau satur jaunu rindu

Pirmās 5 rindas no latviešu faila:
1. Kāds ir jūsu vārds? Šis lauks respondentiem bija neobligāts.
Jānis Berzins.

2. Kādā vecuma grupā jūs esat?
Man ir 45 gadi, un es lielāko daļu savas pieaugušā dzīves esmu dzīvojis un strādājis Rīgā.


In [13]:
# mums varētu būt uzdevums atrast visas rindiņas kuras nav tukšas un nesākas ar ciparu, un izdrukāt to skaitu. To varētu izdarīt ar for ciklu un if nosacījumu, bet tas varētu būt garlaicīgi un neefektīvi, ja mums ir daudz rindu. Labāks veids būtu izmantot list comprehension, lai izveidotu jaunu sarakstu tikai ar tām rindām, kuras atbilst mūsu kritērijiem, un tad izdrukāt šī saraksta garumu. Piemēram:
good_lines = [] # tukšs saraksts (list)
for line in latvian_lines:
    # tādad pārbaudu vai rindā vispār kaut kas ir, un vai tā nesākas ar ciparu
    if line.strip() != '' and not line.lstrip().startswith(tuple('0123456789')):
        print(line, end='')
        good_lines.append(line)

# cik daudz labās līnijas mums ir?
print(f"\nKopā ir {len(good_lines)} labas līnijas, kas nav tukšas un nesākas ar ciparu.")

Jānis Berzins.
Man ir 45 gadi, un es lielāko daļu savas pieaugušā dzīves esmu dzīvojis un strādājis Rīgā.
Vīrietis.
Turpmākai aptaujai varētu izmantot e-pasta adresi janis.berzins45@example.com un tālruņa numuru +371 55550100.
Es strādāju pilnu slodzi kā projektu vadītājs privātā uzņēmumā, un jau 12 gadus man ir stabils profesionāls amats.
Man ir maģistra grāds ekonomikā Latvijas universitātē, un tas man palīdzēja izveidot karjeru ar biznesu un finansēm saistītā darbā.
Es dzīvoju Rīgā kopā ar dzīvesbiedru un vienu pusaudzi, un mēs cenšamies līdzsvarot ikdienas izdevumus ar uzkrājumiem izglītībai un mājokļa uzlabojumiem.
Mani mēneša neto ienākumi ir aptuveni no 2400 līdz 2800 eiro, kas ir virs pilsētas vidējā līmeņa, taču tie joprojām izjūt spiedienu, jo mājokļa, pārtikas, transporta un komunālo pakalpojumu izmaksas ir kļuvušas augstākas.
Lielākais spiediens ir hipotekārā kredīta maksājumi, pārtikas cenas, apkures izmaksas ziemā un kopējais pakalpojumu cenu pieaugums visā pilsētā.
Es pa

In [15]:
from datetime import datetime # iebūvēta biblioteka datuma un laika apstrādei
# atvērsim jaunu fail rakstīšanai
# un ierakstim galveni tajā
# mode="w" nozīmē, ka mēs atveram failu rakstīšanas režīmā, 
# un ja fails jau eksistē, tad tā saturs tiks pārrakstīts. Ja fails neeksistē, tad tas tiks izveidots. encoding="utf-8" nozīmē, ka mēs izmantojam UTF-8 kodējumu, kas atbalsta visus valodas un speciālos simbolus. Ar with konteksta pārvaldnieku mēs varam būt droši, ka fails tiks pareizi aizvērts pēc tam, kad mēs esam pabeiguši ar to strādāt, pat ja rodas kāda kļūda vai izņēmums.
with open("izvada_fails.txt", mode='w', encoding='utf-8') as file:
    file.write("Šī ir galvene jaunajam failam.\n")
    # var izmanot arī print bet tas ir lēnaks
    print("Te ir vēl kāds tekstiņš", file=file) # te būs jauna rinda automatiski
    file.write("Šeit varētu būt citi dati vai rezultāti.\n")
    # laika zīmogs
    file.write(f"Laiks: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
# file šeit jau ir ciet

In [ ]:
# tātad man good_lines jau ir saraksts ar tekstu kuru vēlos pievenot
# bet šoreiz vēlos to pievienot jau esošam failam, nevis pārrakstīt to, 
# tāpēc izmantošu mode='a' (append) režīmu, 
# kas ļauj pievienot jaunu tekstu faila beigās, neizdzēšot esošo saturu. Piemēram:
with open("izvada_fails.txt", mode='a', encoding='utf-8') as file:
    for line in good_lines:
        file.write(line) # moments ka katrs line jau satur \n šinī gadījumā

# kritiski, ka mēs nevaram patvaļīgi failā kaut kur iespraust tekstu, 
# jo faila rakstīšanas režīmā mēs varam tikai pievienot tekstu faila beigās,
#  nevis mainīt vai izdzēst esošo saturu. 
# Ja mēs vēlamies mainīt vai izdzēst esošo saturu, tad mums būtu jālasa viss saturs atpakaļ uz programmu, jāveic nepieciešamās izmaiņas un pēc tam jāraksta viss saturs atpakaļ uz failu. Tas ir svarīgi, lai izvairītos no datu zuduma vai bojājumiem, un lai nodrošinātu, ka mūsu kods ir drošs un efektīvs.

In [18]:
# tagad recepte kā apstrādat lielu failu vienlaicīgi to raksto
# mēs izmantosim ar with atvērt failu rakstīšanas režīmā, un tad izmantosim for ciklu, lai iterētu cauri mūsu datiem (piemēram, good_lines sarakstam), un katru elementu rakstīsim failā ar file.write() metodi. Šādā veidā mēs varam apstrādāt lielu failu vienlaicīgi to raksto, jo mēs neuzglabājam visu saturu atmiņā vienlaicīgi, bet gan rakstām to pa daļām. Piemēram:
with open("../data/day2/survey_response_lv.txt", mode='r', encoding='utf-8') as infile:
    # tagad mums ir infile ir atvērts lasīšanas rēžīma
    # atvērsim arī izvades failu rakstīšanas režīmā
    with open("../data/day2/cleaned_survey_response_lv.txt", mode='w', encoding='utf-8') as outfile:
        # tagad ejam pa rindiņai cauri ienākošajam failam, un rakstām tikai tās rindiņas, kuras nav tukšas un nesākas ar ciparu
        for line in infile: # te svarīgi saprast ka ejam cauri pa rindiņai, viss ienākošais nepaliek atmiņā!
            # un tagad atliek izdomāt loģiku pēc kuras gribam paturēt rindiņu
            # mūsu gadījuma pārbaudam vai rindā vispaŗ kaut kas ir, un vai tā nesākas ar ciparu
            # te varēja izmanto arī regex, bet šis ir vienkāršāks un ātrāks
            if line.strip() != '' and not line.lstrip().startswith(tuple('0123456789')):
                outfile.write(line) # te būs jauna rinda automātiski, jo line jau satur \n
# te jau abi faili ir ciet un darbs ir laimīgi beidzies
# šāda recepte strādās arī uz 2TB faila, jo neglabā atmiņa

In [19]:
# nākošais solis būtu uzrakstīt funkciju
# tā varētu prasīt rakstīt tikai tās rindiņas kurās ir kāds "labais" vārds
def filter_lines_by_keyword(input_path: Path, output_path: Path, keyword: str, encoding: str = 'utf-8') -> None:
    with open(input_path, mode='r', encoding=encoding) as infile:
        with open(output_path, mode='w', encoding=encoding) as outfile:
            for line in infile:
                if keyword in line: # pārbaudam vai atslēgas vārds ir rindā
                    outfile.write(line)

# meklēsims darb mūsu apstrādātajā failā tikai tās rindiņas, kurās ir vārds "darb". Šis vārds varētu būt saistīts ar darba pieredzi, profesiju vai citiem aspektiem, kas varētu būt interesanti analizēt. Piemēram:
filter_lines_by_keyword("../data/day2/cleaned_survey_response_lv.txt", 
                        "darba_rindas.txt", # lieku tepat notepad lai varētu redzēt rezultātu, bet varētu arī datu mapē
                        keyword="darb") # mums nevajag rakstīt utf-8 jo tas jau ir noklusējuma kodējums mūsu funkcijā

In [22]:
# ja ideja saprasta, tad nākoša funkcija varētu būt līdzīga
# bet pārbaudītu prefix
# parbaudīt vairākus keyword
# un arī postfix
def filter_lines_by_prefix_keywords_postfix(input_path: Path, 
                                            output_path: Path, 
                                            prefix: str = "", 
                                            keywords: tuple[str] = (), # noklusētam labāk būt tuple
                                            postfix: str = "", 
                                            encoding: str = 'utf-8') -> None:
    with open(input_path, mode='r', encoding=encoding) as infile:
        with open(output_path, mode='w', encoding=encoding) as outfile:
            for line in infile:
                if line.startswith(prefix) and any(keyword in line for keyword in keywords) and line.endswith(postfix):
                    outfile.write(line)

# pārbaudam tukšo variantu - vajadzētu sanākt kopijai :)
# ievērojam ja izlaižam kādu noklusēto parametru
# tad padotajam ir jānorāda nosaukums, piemēram, prefix="", jo citādi Python nesapratīs, ka mēs gribam izmantot noklusēto vērtību, nevis padot to kā pozicionālo argumentu
filter_lines_by_prefix_keywords_postfix("../data/day2/cleaned_survey_response_lv.txt", 
                                        "janis_darb.txt",
                                        keywords=("Jānis", "darb")) # šeit mēs meklējam rindiņas, kurās ir vārds "Jānis" vai "darb", un nav svarīgi kāds ir prefix un postfix, jo mēs izmantojam noklusētās vērtības)


In [ ]:
# tātad ja mākam apstrādat vienu failu
# tad mācēsim apstrādat arī citus līdzīgus failus
# rīt tātad apstrādāsim vairākkus failus un
# izveidosim CSV failu no atbildēm kas apkopo vairākus teksta failus vienlaicīgi.

In [ ]:
def inspect_text_file(path: Path = RAW_PATH) -> dict[str, object]:
    """TODO: salikt kopā pilnu pārbaudes plūsmu."""
    summary = {
        'path': str(path),
        'total_lines': None,
        'blank_lines': None,
        'digit_lines': None,
        'extra_space_lines': None,
        'duplicate_lines': None,
    }
    raise NotImplementedError


# TODO:
# inspection_summary = inspect_text_file()
# inspection_summary


## Teksta rindu tīrīšana

Mērķis: normalizēt rindas un saglabāt tīrītu tekstu jaunā failā.

Ievades fails:
- `data/day2/responses_raw.txt`

Sagaidāmā izvade:
- Python objekts: `list[str]`
- Izvades fails: `data/day2/generated_cleaned_responses.txt`


In [ ]:
def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    """TODO: nolasīt failu un atgriezt rindu sarakstu."""
    raise NotImplementedError


def clean_line(line: str) -> str:
    """TODO: normalizēt vienu teksta rindu."""
    # Ieteicamā secība: lower -> strip -> remove punctuation -> normalize spaces
    raise NotImplementedError


def clean_lines(lines: list[str]) -> list[str]:
    """TODO: iztīrīt visas rindas un atmest tukšos rezultātus."""
    raise NotImplementedError


def write_lines(path: Path, lines: list[str], encoding: str = 'utf-8') -> None:
    """TODO: saglabāt rindas failā."""
    raise NotImplementedError


def compare_raw_and_cleaned(raw_lines: list[str], cleaned_lines: list[str], limit: int = 5) -> None:
    """TODO: izdrukāt dažus RAW un CLEAN piemērus."""
    raise NotImplementedError


In [ ]:
def clean_text_lines(
    raw_path: Path = RAW_PATH,
    output_path: Path = GENERATED_CLEANED_PATH,
) -> list[str]:
    """TODO: izpildīt pilnu rindu tīrīšanas plūsmu."""
    # Rezultāts: list[str] un saglabāts output_path failā.
    raise NotImplementedError


# TODO:
# cleaned_lines_output = clean_text_lines()
# cleaned_lines_output[:5]


## Vārdu biežuma pārskata izveide

Mērķis: no iztīrītajām rindām izveidot vārdu biežuma tabulu, izņemot stopvārdus.

Ievades faili:
- `data/day2/generated_cleaned_responses.txt`
- `data/day2/stopwords_lv.txt`

Sagaidāmā izvade:
- Python objekts: `list[tuple[str, int]]`
- Izvades fails: `data/day2/generated_word_frequencies.txt`


In [ ]:
def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    """TODO: nolasīt failu kā rindu sarakstu."""
    raise NotImplementedError


def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    """TODO: ielādēt stopvārdus kopā."""
    raise NotImplementedError


def tokenize_lines(lines: list[str]) -> list[str]:
    """TODO: sadalīt rindas token sarakstā."""
    raise NotImplementedError


def remove_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
    """TODO: izņemt stopvārdus."""
    raise NotImplementedError


def count_words(tokens: list[str]) -> dict[str, int]:
    """TODO: saskaitīt token biežumus."""
    raise NotImplementedError


def sort_word_counts(word_counts: dict[str, int]) -> list[tuple[str, int]]:
    """TODO: sakārtot pēc biežuma dilstoši un tad alfabētiski."""
    raise NotImplementedError


def write_word_frequencies(path: Path, word_counts: list[tuple[str, int]], encoding: str = 'utf-8') -> None:
    """TODO: saglabāt failā formāts vārds<TAB>skaits."""
    raise NotImplementedError


In [ ]:
def build_word_frequency_report(
    cleaned_path: Path = GENERATED_CLEANED_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    output_path: Path = GENERATED_FREQ_PATH,
) -> list[tuple[str, int]]:
    """TODO: izveidot pilnu vārdu biežuma pārskata plūsmu."""
    # cleaned_lines -> tokens -> filtered_tokens -> word_counts -> sorted_counts
    raise NotImplementedError


# TODO:
# word_frequency_report = build_word_frequency_report()
# word_frequency_report[:10]


## Refaktorēšana ar comprehensions un generators

Mērķis: to pašu plūsmu pārrakstīt kompaktāk ar comprehensions un generator pieeju.

Ievades faili:
- `data/day2/responses_raw.txt`
- `data/day2/stopwords_lv.txt`

Sagaidāmā izvade:
- Python objekts: `dict[str, object]`
- Izvades faili:
  - `data/day2/generated_cleaned_responses_v2.txt`
  - `data/day2/generated_word_frequencies_v2.txt`


In [ ]:
def clean_line(line: str) -> str:
    """TODO: izmantot iepriekšējo rindas tīrīšanas loģiku."""
    raise NotImplementedError


def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    """TODO: nolasīt faila rindas."""
    raise NotImplementedError


def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    """TODO: ielādēt stopvārdus ar comprehension."""
    raise NotImplementedError


def cleaned_lines_from_file(path: Path, encoding: str = 'utf-8') -> list[str]:
    """TODO: nolasīt un iztīrīt rindas ar list comprehension."""
    raise NotImplementedError


In [ ]:
def cleaned_line_generator(path: Path, encoding: str = 'utf-8'):
    """TODO: atgriezt iztīrītās netukšās rindas pa vienai."""
    raise NotImplementedError


def token_generator(lines: list[str]):
    """TODO: ģenerēt token pa vienam no katras rindas."""
    raise NotImplementedError


def build_word_counts(tokens: list[str], stopwords: set[str]) -> dict[str, int]:
    """TODO: izveidot biežumu vārdnīcu filtrētajiem token."""
    raise NotImplementedError


def summarize_counts(word_counts: dict[str, int], top_n: int = 10) -> list[tuple[str, int]]:
    """TODO: atgriezt top N biežākos vārdus."""
    raise NotImplementedError


def write_lines(path: Path, lines: list[str], encoding: str = 'utf-8') -> None:
    """TODO: saglabāt teksta rindas failā."""
    raise NotImplementedError


def write_word_frequencies(path: Path, pairs: list[tuple[str, int]], encoding: str = 'utf-8') -> None:
    """TODO: saglabāt vārdu-skaita pārus failā."""
    raise NotImplementedError


In [ ]:
def run_comprehension_pipeline(
    raw_path: Path = RAW_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    cleaned_output_path: Path = GENERATED_CLEANED_V2_PATH,
    frequency_output_path: Path = GENERATED_FREQ_V2_PATH,
) -> dict[str, object]:
    """TODO: izpildīt kompaktāku pipeline versiju."""
    # Sagaidāmās atslēgas: cleaned_lines, tokens, word_counts, top_words
    raise NotImplementedError


# TODO:
# comprehension_pipeline_results = run_comprehension_pipeline()
# comprehension_pipeline_results['top_words']


## Pilnas plūsmas iepakošana klasē

Mērķis: apvienot ielādi, tīrīšanu, tokenizāciju un saglabāšanu vienā atkārtoti lietojamā klasē.

Ievades faili:
- `data/day2/responses_raw.txt`
- `data/day2/stopwords_lv.txt`

Sagaidāmā izvade:
- Python objekts: `dict[str, object]`
- Izvades faili:
  - `data/day2/generated_cleaned_responses_final.txt`
  - `data/day2/generated_word_frequencies_final.txt`


In [ ]:
def clean_line(line: str) -> str:
    """TODO: izmantot iepriekš izveidoto normalizācijas pieeju."""
    raise NotImplementedError


class TextCorpus:
    """TODO: neliels atkārtoti lietojams objekts teksta ielādei, tīrīšanai un analīzei."""

    def __init__(self, raw_path: Path, stopwords_path: Path, encoding: str = 'utf-8'):
        self.raw_path = raw_path
        self.stopwords_path = stopwords_path
        self.encoding = encoding
        self.raw_lines: list[str] = []
        self.stopwords: set[str] = set()
        self.cleaned_lines: list[str] = []
        self.tokens: list[str] = []

    def load(self) -> None:
        """TODO: ielādēt neapstrādātās rindas un stopvārdus."""
        raise NotImplementedError

    def clean(self) -> None:
        """TODO: iztīrīt neapstrādātās rindas."""
        raise NotImplementedError

    def tokenize(self) -> None:
        """TODO: sadalīt iztīrītās rindas token sarakstā."""
        raise NotImplementedError

    def filtered_tokens(self) -> list[str]:
        """TODO: atgriezt token bez stopvārdiem."""
        raise NotImplementedError

    def word_counts(self) -> dict[str, int]:
        """TODO: atgriezt filtrēto token biežumus."""
        raise NotImplementedError

    def top_words(self, n: int = 10) -> list[tuple[str, int]]:
        """TODO: atgriezt top N vārdus pēc biežuma."""
        raise NotImplementedError

    def save_cleaned(self, path: Path) -> None:
        """TODO: saglabāt iztīrītās rindas failā."""
        raise NotImplementedError

    def save_word_frequencies(self, path: Path) -> None:
        """TODO: saglabāt sakārtotus vārdu biežumus failā."""
        raise NotImplementedError

    def summary(self) -> dict[str, int]:
        """TODO: atgriezt kompaktu kopsavilkumu."""
        raise NotImplementedError


In [ ]:
def run_text_corpus_pipeline(
    raw_path: Path = RAW_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    cleaned_output_path: Path = GENERATED_CLEANED_FINAL_PATH,
    frequency_output_path: Path = GENERATED_FREQ_FINAL_PATH,
) -> dict[str, object]:
    """TODO: izpildīt pilno klases pieeju teksta apstrādes plūsmai."""
    raise NotImplementedError


# TODO:
# text_corpus_results = run_text_corpus_pipeline()
# text_corpus_results['summary']


## Aptaujas failu pārveidošana par CSV

Mērķis: vairākus teksta aptaujas failus pārvērst strukturētās CSV rindās.

Ievades faili:
- `data/day2/surveys/*.txt`
- `data/day2/stopwords_lv.txt`

Sagaidāmā izvade:
- Python objekts: `list[dict[str, str | int | None]]`
- Izvades fails: `data/day2/surveys/survey_responses_summary.csv`


In [ ]:
DEFAULT_OUTPUT_NAME = 'survey_responses_summary.csv'

QUESTION_FIELD_MAP = {
    1: 'name',
    2: 'age_answer',
    3: 'gender',
    4: 'contact_answer',
    5: 'employment_answer',
    6: 'education_answer',
    7: 'household_answer',
    8: 'income_answer',
    9: 'economic_pressures_answer',
    10: 'drive_answer',
    11: 'bike_answer',
    12: 'inflation_impact_answer',
    13: 'job_finance_security_answer',
    14: 'riga_economy_view_answer',
    15: 'improvements_answer',
}

TEXT_FIELDS_FOR_KEYWORDS = [
    'employment_answer',
    'education_answer',
    'household_answer',
    'income_answer',
    'economic_pressures_answer',
    'inflation_impact_answer',
    'job_finance_security_answer',
    'riga_economy_view_answer',
    'improvements_answer',
]

CSV_COLUMNS = [
    'source_file', 'name', 'age', 'age_answer', 'gender', 'email', 'phone',
    'employment_type', 'occupation', 'years_in_profession', 'employment_answer',
    'employment_answer_keywords', 'education_answer', 'education_answer_keywords',
    'household_answer', 'household_answer_keywords', 'income_min_eur', 'income_max_eur',
    'income_answer', 'income_answer_keywords', 'economic_pressures_answer',
    'economic_pressures_answer_keywords', 'drive_km_week', 'bike_km_week',
    'inflation_impact_answer', 'inflation_impact_answer_keywords',
    'job_finance_security_answer', 'job_finance_security_answer_keywords',
    'riga_economy_view_answer', 'riga_economy_view_answer_keywords',
    'improvements_answer', 'improvements_answer_keywords',
]


In [ ]:
def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    """TODO: ielādēt stopvārdus no faila."""
    raise NotImplementedError


def read_text(path: Path, encoding: str = 'utf-8') -> str:
    """TODO: nolasīt visu teksta failu."""
    raise NotImplementedError


def parse_survey_blocks(text: str) -> dict[int, str]:
    """TODO: ar regex palīdzību izvilkt numurētos jautājumu-atbilžu blokus."""
    raise NotImplementedError


def normalize_text(text: str) -> str:
    """TODO: pārvērst tekstu mazajos burtos un sakārtot simbolus keyword ieguvei."""
    raise NotImplementedError


def keyword_text(text: str, stopwords: set[str]) -> str:
    """TODO: iegūt normalizētus atslēgvārdus bez stopvārdiem."""
    raise NotImplementedError


def compact_value(text: str) -> str:
    """TODO: noņemt liekās atstarpes un nevajadzīgas beigu pieturzīmes."""
    raise NotImplementedError


def first_int(text: str) -> int | None:
    """TODO: atrast pirmo veselo skaitli tekstā."""
    raise NotImplementedError


def income_range(text: str) -> tuple[int | None, int | None]:
    """TODO: izvilkt minimālo un maksimālo ienākumu vērtību."""
    raise NotImplementedError


def extract_email(text: str) -> str:
    """TODO: izvilkt e-pasta adresi, ja tāda ir."""
    raise NotImplementedError


def extract_phone(text: str) -> str:
    """TODO: izvilkt tālruņa numuru, ja tāds ir."""
    raise NotImplementedError


def employment_type(text: str) -> str:
    """TODO: noteikt nodarbinātības veidu no atbildes."""
    raise NotImplementedError


def occupation(text: str) -> str:
    """TODO: izvilkt profesiju vai lomu no nodarbinātības apraksta."""
    raise NotImplementedError


def write_csv(path: Path, rows: list[dict[str, str | int | None]], encoding: str = 'utf-8') -> None:
    """TODO: saglabāt strukturētos datus CSV failā ar noteikto kolonnu secību."""
    raise NotImplementedError


In [ ]:
def survey_row(path: Path, stopwords: set[str]) -> dict[str, str | int | None]:
    """TODO: pārvērst vienu aptaujas failu vienā CSV rindas vārdnīcā."""
    # Ieteikums: izlasi failu, saparsē atbildes, aizpildi row un atvasinātos laukus.
    raise NotImplementedError


In [ ]:
def survey_folder_to_csv(
    input_folder: Path = SURVEYS_FOLDER,
    output_path: Path | None = None,
    stopwords_path: Path = STOPWORDS_PATH,
) -> list[dict[str, str | int | None]]:
    """TODO: apstrādāt visu aptaujas mapi un saglabāt rezultātu CSV failā."""
    # Sagaidāmie soļi: pārbaudīt mapi, ielādēt stopvārdus, atrast *.txt failus,
    # izveidot survey_row katram failam, saglabāt CSV un atgriezt rows.
    raise NotImplementedError


# TODO:
# survey_rows = survey_folder_to_csv()
# survey_rows[:2]
